# Stage 4B Baseline Experiments — Final Dataset (N = 800)

This notebook processes the **final** curated subset.
The pilot version is in `stage4b_baseline_experiments__pilot.ipynb`.
Inputs: `datasets/detector_processed/final_detector_scores.csv`, `datasets/emotion_annotated/metadata/final_video_emotion_features.csv`
Outputs: `datasets/metadata/final_xception_*.csv`

# Stage 4 — Baseline + Merge + Experiments


## 1. Load data

In [2]:

import pandas as pd
from pathlib import Path

ROOT = Path("../")

emotion_path = ROOT / "datasets/emotion_annotated/metadata/final_video_emotion_features.csv"
manifest_path = ROOT / "exports/final/csv/final_manifest_export.csv"
detector_path = ROOT / "datasets/detector_processed/final_detector_scores.csv"

emotion_df = pd.read_csv(emotion_path)
manifest_df = pd.read_csv(manifest_path)
detector_df = pd.read_csv(detector_path)

print(len(emotion_df), len(manifest_df), len(detector_df))


790 800 790


## 2. Merge

In [3]:

# Only carry needed columns from detector_df to avoid duplicate label/split columns
detector_cols = ["video_id", "detector_score", "detector_pred"]

# Drop duplicate columns from emotion_df before merging
emotion_drop = [c for c in emotion_df.columns if c in manifest_df.columns and c != "video_id"]
df = manifest_df.merge(emotion_df.drop(columns=emotion_drop), on="video_id")
df = df.merge(detector_df[detector_cols], on="video_id")

# Encode label: "fake" → 1, "real" → 0
df["label"] = (df["label"].str.lower() == "fake").astype(int)

print("Shape:", df.shape)
print("Label distribution:\n", df["label"].value_counts())
df.head()


Shape: (790, 73)
Label distribution:
 label
0    400
1    390
Name: count, dtype: int64


,video_id,path,relative_path,split,label,source_subset,manipulation_family,manipulation_type,identity,source_identity,...,mean_score_relief,mean_score_sadness,mean_score_sexual_lust,mean_score_shame,mean_score_sourness,mean_score_teasing,mean_score_thankfulness_gratitude,mean_score_triumph,detector_score,detector_pred
0,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,videos/fake/TalkingFace/id35_0005_test_id07802...,Celeb-synthesis/TalkingFace/EDTalk/id35_0005_t...,train,1,Celeb-synthesis,TalkingFace,EDTalk,id35,id35,...,0.574881,-0.134643,1.015455,-0.337185,-0.657182,0.792035,1.587619,-0.916439,0.621137,1
1,Celeb-synthesis__TalkingFace__AniTalker__id48_...,videos/fake/TalkingFace/id48_0002_test_id04656...,Celeb-synthesis/TalkingFace/AniTalker/id48_000...,train,1,Celeb-synthesis,TalkingFace,AniTalker,id48,id48,...,0.285548,-0.148196,2.320999,-0.139389,-0.407562,1.057906,2.598602,-0.877789,0.800077,1
2,Celeb-real__id50_0005__3870d23c,videos/real/Celeb-real/id50_0005.mp4,Celeb-real/id50_0005.mp4,test,0,Celeb-real,NaN,Celeb-real,id50,id50,...,0.697492,-0.317364,1.225383,-0.432930,-0.560299,0.632307,1.407119,-0.901347,0.739608,1
3,Celeb-synthesis__FaceReenact__TPSMM__id23_id19...,videos/fake/FaceReenact/id23_id19_0000.mp4,Celeb-synthesis/FaceReenact/TPSMM/id23_id19_00...,test,1,Celeb-synthesis,FaceReenact,TPSMM,id23,id23,...,0.241701,-0.222058,1.685264,-0.440217,-0.736206,0.933438,1.848022,-0.586169,0.725605,1
4,Celeb-real__id20_0001__5e2f4954,videos/real/Celeb-real/id20_0001.mp4,Celeb-real/id20_0001.mp4,train,0,Celeb-real,NaN,Celeb-real,id20,id20,...,0.248174,0.147449,0.358857,0.022838,-0.473432,0.173443,1.033999,-0.921456,0.993916,1


## 3. Metrics (baseline)

In [4]:

import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

y_true = df["label"].values
y_score = df["detector_score"].values
y_pred = (y_score >= 0.5).astype(int)

auc = roc_auc_score(y_true, y_score)
acc = accuracy_score(y_true, y_pred)
f1  = f1_score(y_true, y_pred)

# Bootstrap 95% CI for AUC
rng = np.random.default_rng(42)
boot_aucs = [
    roc_auc_score(y_true[idx := rng.integers(0, len(y_true), len(y_true))], y_score[idx])
    for _ in range(2000)
]
auc_lo, auc_hi = np.percentile(boot_aucs, [2.5, 97.5])

print(f"AUC : {auc:.4f}  (95% CI {auc_lo:.4f}–{auc_hi:.4f})")
print(f"ACC : {acc:.4f}")
print(f"F1  : {f1:.4f}")


AUC : 0.5396  (95% CI 0.4992–0.5790)
ACC : 0.4861
F1  : 0.6206


## 4. Emotion analysis

In [5]:

df["arousal_bin"] = pd.qcut(df["mean_arousal"], 3, labels=["low", "mid", "high"])

rng = np.random.default_rng(42)

print(f"{'Bin':<6}  {'AUC':>6}  {'95% CI':>18}  {'n':>4}")
print("-" * 42)
for g in ["low", "mid", "high"]:
    sub = df[df["arousal_bin"] == g]
    yt, ys = sub["label"].values, sub["detector_score"].values
    auc = roc_auc_score(yt, ys)
    boot = [
        roc_auc_score(yt[idx := rng.integers(0, len(yt), len(yt))], ys[idx])
        for _ in range(2000)
    ]
    lo, hi = np.percentile(boot, [2.5, 97.5])
    print(f"{str(g):<6}  {auc:.4f}  ({lo:.4f} – {hi:.4f})  {len(sub):>4}")


Bin        AUC              95% CI     n
------------------------------------------
low     0.5452  (0.4748 – 0.6141)   264
mid     0.5182  (0.4475 – 0.5885)   263
high    0.5177  (0.4484 – 0.5875)   263


## 5. Fusion model

In [6]:

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

features = [
    "detector_score",
    "mean_arousal",
    "std_arousal",
    "mean_valence",
    "emotion_entropy",
    "transition_rate",
]

X = df[features].fillna(0).values
y = df["label"].values

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fusion_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])

fusion_proba = cross_val_predict(fusion_pipe, X, y, cv=cv, method="predict_proba")[:, 1]
fusion_auc = roc_auc_score(y, fusion_proba)

rng = np.random.default_rng(42)
boot = [
    roc_auc_score(y[idx := rng.integers(0, len(y), len(y))], fusion_proba[idx])
    for _ in range(2000)
]
lo, hi = np.percentile(boot, [2.5, 97.5])

print(f"Fusion AUC (5-fold CV): {fusion_auc:.4f}  (95% CI {lo:.4f}–{hi:.4f})")


Fusion AUC (5-fold CV): 0.6569  (95% CI 0.6199–0.6955)


## 6. Compare

In [7]:

# ── Detector baseline (no model needed — direct scores) ──────────────────────
baseline_auc = roc_auc_score(y, df["detector_score"].values)

# ── Emotion-only (5-fold CV, same pipeline) ───────────────────────────────────
emotion_features = [
    "mean_arousal", "std_arousal", "mean_valence",
    "emotion_entropy", "transition_rate",
]
X_emo = df[emotion_features].fillna(0).values

emotion_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])
emotion_proba = cross_val_predict(emotion_pipe, X_emo, y, cv=cv, method="predict_proba")[:, 1]
emotion_auc = roc_auc_score(y, emotion_proba)

rng2 = np.random.default_rng(42)
def _boot_auc(yt, ys, n=2000):
    rng_ = np.random.default_rng(42)
    b = [roc_auc_score(yt[i := rng_.integers(0, len(yt), len(yt))], ys[i]) for _ in range(n)]
    return np.percentile(b, [2.5, 97.5])

b_lo, b_hi = _boot_auc(y, df["detector_score"].values)
e_lo, e_hi = _boot_auc(y, emotion_proba)
f_lo, f_hi = lo, hi  # from fusion cell above

print(f"{'Model':<20}  {'AUC':>6}  {'95% CI':>18}")
print("-" * 50)
print(f"{'Detector only':<20}  {baseline_auc:.4f}  ({b_lo:.4f} – {b_hi:.4f})")
print(f"{'Emotion only (CV)':<20}  {emotion_auc:.4f}  ({e_lo:.4f} – {e_hi:.4f})")
print(f"{'Fusion (CV)':<20}  {fusion_auc:.4f}  ({f_lo:.4f} – {f_hi:.4f})")


Model                    AUC              95% CI
--------------------------------------------------
Detector only         0.5396  (0.4992 – 0.5790)
Emotion only (CV)     0.6595  (0.6221 – 0.6973)
Fusion (CV)           0.6569  (0.6199 – 0.6955)


## 7. Robust evaluation (train->test + GroupKFold + significance)

This section adds a leakage-aware evaluation:
- final holdout by `split` (train -> test)
- GroupKFold on train grouped by `identity`
- permutation test for AUC difference on test


In [8]:
import numpy as np
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# --- Split holdout (train -> test)
split_col = df['split'].astype(str).str.lower()
train_df = df[split_col == 'train'].copy()
test_df = df[split_col == 'test'].copy()

assert len(train_df) > 0 and len(test_df) > 0, 'train/test split is empty in df'
assert train_df['label'].nunique() == 2 and test_df['label'].nunique() == 2, 'Both classes must exist in train and test'

fusion_features = [
    'detector_score',
    'mean_arousal',
    'std_arousal',
    'mean_valence',
    'emotion_entropy',
    'transition_rate',
]
emotion_features = [
    'mean_arousal',
    'std_arousal',
    'mean_valence',
    'emotion_entropy',
    'transition_rate',
]

X_train_fusion = train_df[fusion_features].fillna(0).values
X_test_fusion = test_df[fusion_features].fillna(0).values
X_train_emo = train_df[emotion_features].fillna(0).values
X_test_emo = test_df[emotion_features].fillna(0).values
y_train = train_df['label'].values
y_test = test_df['label'].values

def make_pipe():
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000)),
    ])

# --- Holdout metrics
det_test_scores = test_df['detector_score'].values
det_test_auc = roc_auc_score(y_test, det_test_scores)

emo_model = make_pipe().fit(X_train_emo, y_train)
emo_test_scores = emo_model.predict_proba(X_test_emo)[:, 1]
emo_test_auc = roc_auc_score(y_test, emo_test_scores)

fusion_model = make_pipe().fit(X_train_fusion, y_train)
fusion_test_scores = fusion_model.predict_proba(X_test_fusion)[:, 1]
fusion_test_auc = roc_auc_score(y_test, fusion_test_scores)

# --- GroupKFold on train (grouped by identity to reduce identity leakage)
group_keys = train_df['identity'].fillna(train_df['video_id']).astype(str).values
n_unique_groups = len(np.unique(group_keys))
n_splits = min(5, n_unique_groups)
assert n_splits >= 2, 'Need at least 2 unique groups for GroupKFold'
gkf = GroupKFold(n_splits=n_splits)

emo_oof = cross_val_predict(make_pipe(), X_train_emo, y_train, cv=gkf, groups=group_keys, method='predict_proba')[:, 1]
fusion_oof = cross_val_predict(make_pipe(), X_train_fusion, y_train, cv=gkf, groups=group_keys, method='predict_proba')[:, 1]

emo_gkf_auc = roc_auc_score(y_train, emo_oof)
fusion_gkf_auc = roc_auc_score(y_train, fusion_oof)

def bootstrap_auc_ci(y_true, y_score, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    aucs = []
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y_true[idx])) < 2:
            continue
        aucs.append(roc_auc_score(y_true[idx], y_score[idx]))
    lo, hi = np.percentile(aucs, [2.5, 97.5])
    return float(lo), float(hi)

def paired_auc_permutation_test(y_true, s1, s2, n_perm=5000, seed=42):
    y_true = np.asarray(y_true)
    s1 = np.asarray(s1)
    s2 = np.asarray(s2)
    obs = roc_auc_score(y_true, s1) - roc_auc_score(y_true, s2)
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_perm):
        swap = rng.random(len(y_true)) < 0.5
        p1 = np.where(swap, s2, s1)
        p2 = np.where(swap, s1, s2)
        diffs.append(roc_auc_score(y_true, p1) - roc_auc_score(y_true, p2))
    diffs = np.asarray(diffs)
    p = (np.sum(np.abs(diffs) >= abs(obs)) + 1) / (len(diffs) + 1)
    return float(obs), float(p)

det_lo, det_hi = bootstrap_auc_ci(y_test, det_test_scores)
emo_lo, emo_hi = bootstrap_auc_ci(y_test, emo_test_scores)
fus_lo, fus_hi = bootstrap_auc_ci(y_test, fusion_test_scores)
delta_auc, p_value = paired_auc_permutation_test(y_test, fusion_test_scores, emo_test_scores)

print(f'Train size: {len(train_df):,} | Test size: {len(test_df):,}')
print(f'GroupKFold groups (identity/video_id fallback): {n_unique_groups:,}, folds={n_splits}')
print()
print('Holdout test AUC (trained on train only):')
print(f'  Detector only   : {det_test_auc:.4f} ({det_lo:.4f}-{det_hi:.4f})')
print(f'  Emotion only    : {emo_test_auc:.4f} ({emo_lo:.4f}-{emo_hi:.4f})')
print(f'  Fusion          : {fusion_test_auc:.4f} ({fus_lo:.4f}-{fus_hi:.4f})')
print()
print('GroupKFold (train only, grouped by identity) AUC:')
print(f'  Emotion only    : {emo_gkf_auc:.4f}')
print(f'  Fusion          : {fusion_gkf_auc:.4f}')
print()
print('Permutation test on test set (Fusion vs Emotion):')
print(f'  dAUC = {delta_auc:+.4f}, p-value = {p_value:.4f}')


Train size: 570 | Test size: 155
GroupKFold groups (identity/video_id fallback): 181, folds=5

Holdout test AUC (trained on train only):
  Detector only   : 0.6029 (0.5153-0.6871)
  Emotion only    : 0.7139 (0.6310-0.7932)
  Fusion          : 0.7140 (0.6313-0.7933)

GroupKFold (train only, grouped by identity) AUC:
  Emotion only    : 0.6524
  Fusion          : 0.6510

Permutation test on test set (Fusion vs Emotion):
  dAUC = +0.0002, p-value = 0.8636
